# Get all mediums per artist
I wanted to extract both top mediums and all mediums for each artist to explore the data a little more, ended up using this to draw the individual artist svgs used on my final website - mediums were mapped to brush types and filled in hand drawn circles for each artist based on their mediums so that their point's is an abstract representation of their work

<img src="../../images/artist_dots.png" width="400">

In [1]:
import pandas as pd

In [2]:
day = "04-25"
artworks_path = f"../../data/downloaded/processed/{day}/artworks.csv"

In [ ]:
df = pd.read_csv(artworks_path, dtype={"artist": "string"}, keep_default_na=False)
df["artist"] = df["artist"].fillna("").astype(str).str.strip()
df = df[df["artist"] != ""].copy()

parsed_lists = (
    df["mediums_parsed"]
    .fillna("")
    .astype(str)
    .str.findall(r"'([^']+)'|\"([^\"]+)\"")
    .apply(lambda pairs: [(a or b).strip() for a, b in pairs if (a or b).strip()])
)

df["medium_list"] = parsed_lists

exploded = df.explode("medium_list")

counts = (
    exploded.groupby(["artist", "medium_list"])
    .size()
    .reset_index(name="count")
    .sort_values(["artist", "count", "medium_list"], ascending=[True, False, True])
)

top3 = counts.groupby("artist", group_keys=False).head(3)

summary = (
    top3.groupby("artist")
    .agg(
        top_3_mediums=("medium_list", list),
        top_3_counts=("count", list),
        avg_top_3_count=("count", "mean"),
    )
    .reset_index()
)
summary["avg_top_3_count"] = summary["avg_top_3_count"].round(2)

top3_mediums_by_artist = {
    row["artist"]: {
        "top_3_mediums": list(zip(row["top_3_mediums"], row["top_3_counts"])),
        "avg_top_3_count": float(row["avg_top_3_count"]),
    }
    for _, row in summary.iterrows()
}

In [13]:
top3_mediums_by_artist

{'0': {'top_3_mediums': [('M06', 48), ('M03', 11), ('M11', 7)],
  'avg_top_3_count': 22.0},
 '1': {'top_3_mediums': [('M02', 1)], 'avg_top_3_count': 1.0},
 '10': {'top_3_mediums': [('M04', 5), ('M03', 4), ('M02', 3)],
  'avg_top_3_count': 4.0},
 '11': {'top_3_mediums': [('M06', 5)], 'avg_top_3_count': 5.0},
 '12': {'top_3_mediums': [('M04', 4), ('M03', 2), ('M02', 1)],
  'avg_top_3_count': 2.33},
 '13': {'top_3_mediums': [('M06', 4), ('M05', 3), ('M02', 2)],
  'avg_top_3_count': 3.0},
 '14': {'top_3_mediums': [('M04', 2), ('M03', 1)], 'avg_top_3_count': 1.5},
 '15': {'top_3_mediums': [('M05', 8), ('M02', 1)], 'avg_top_3_count': 4.5},
 '16': {'top_3_mediums': [('M14', 1)], 'avg_top_3_count': 1.0},
 '17': {'top_3_mediums': [('M03', 1)], 'avg_top_3_count': 1.0},
 '18': {'top_3_mediums': [('M02', 1)], 'avg_top_3_count': 1.0},
 '19': {'top_3_mediums': [('M03', 1), ('M05', 1)], 'avg_top_3_count': 1.0},
 '2': {'top_3_mediums': [('M02', 4), ('M05', 2)], 'avg_top_3_count': 3.0},
 '20': {'top_3_

In [11]:
artists_path = f"../../data/downloaded/processed/{day}/artists.csv"

artworks_df = pd.read_csv(artworks_path, dtype={"artist": "string"}, keep_default_na=False)
artists_df = pd.read_csv(artists_path, dtype={"id": "string"})

raw_mediums_df = (
    artworks_df.groupby("artist", sort=True)["medium"]
    .agg(lambda s: ", ".join(pd.unique(s)))
    .reset_index(name="all_raw_mediums")
    .rename(columns={"artist": "id"})
)

artist_names = artists_df[["id", "name"]].copy()
artist_mediums_df = artist_names.merge(raw_mediums_df, on="id", how="left")
artist_mediums_df["all_raw_mediums"] = artist_mediums_df["all_raw_mediums"].fillna("")

artist_mediums_df.head(20)

,id,name,all_raw_mediums
0,0,Isamu Noguchi,Laminated and carved avodire | Design | Archit...
1,1,Amanda Ross-Ho,Two chromogenic prints | Photograph
2,2,Laurel Nakadate,"Three-channel video (color, sound) | Installat..."
3,3,Mequitta Ahuja,Oil on canvas | Paintings
4,4,Kip Fulbeck,Performance readings | multiple image performance
5,5,Ruth Asawa,"Lithograph | Print | Drawings & Prints, One fr..."
6,6,Do Ho Suh,"Inkjet print | Prints, Fiber-tipped pen, and b..."
7,7,Oscar yi Hou,Oil on canvas | Paintings
8,8,Sarah Sze,"Lithograph | Print | Drawings & Prints, Lithog..."
9,9,Wu Tsang,"High-definition video (color, sound) | Video |..."


In [8]:
artist_mediums_df.to_csv("../../data/addl/artist_mediums.csv", index=False)